# Лаборатория 9. Экономим на истории диалога

**Что мы сделаем:** проведём с живой моделью один и тот же разговор четырьмя способами
и на каждом ходе посмотрим на число, за которое на самом деле платят, — `prompt_tokens`.

| Шаг | Что узнаем |
|---|---|
| 1 | Модель правда ничего не помнит — проверим своими глазами |
| 2 | Полная история: как растёт запрос от хода к ходу |
| 3 | Скользящее окно: дёшево, но что оно забудет? |
| 4 | Пересказ: модель сама сжимает старое |
| 5 | Долгая память: факты выписываются отдельно |
| 6 | Сравнение всех способов одной таблицей |
| 7 | Кэш начала запроса: скидка, для которой ничего не надо забывать |

**Что понадобится:** код класса от учителя.

**Сколько запросов:** около 70. Лимит класса — 120 в час, поэтому не запускай весь
ноутбук по кругу и не запускай сразу после другого большого ноутбука: при подготовке
лаборатории именно так и случилась ошибка 429 «слишком много запросов» (тема 11).
Лучше перезапускай отдельные шаги.

In [ ]:
!pip -q install openai

In [ ]:
import getpass
import json
import os
import re
from pprint import pprint

from openai import OpenAI

ADRES = "https://ai9.adelfos.ru/api/v1"
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata
    KOD_KLASSA = userdata.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD") or getpass.getpass("Код класса: ")

client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
print("Подключились.")

## Подготовка: функция, которая показывает всё

Во всех шагах мы будем звать модель через одну функцию. Она не только возвращает ответ,
но и печатает то, что обычно спрятано:

* **сколько сообщений** ушло модели и какой они длины;
* **`prompt_tokens`** — сколько токенов модель прочитала (за это платим по цене входа);
* **`completion_tokens`** — сколько написала (по цене выхода).

Эти числа присылает сервер вместе с ответом, в поле `usage`. Мы их не угадываем.

In [ ]:
def sprosit(soobshcheniya, pokazat_zapros=False, max_tokens=150, **dop):
    """Отправляет сообщения модели. Возвращает (текст ответа, usage)."""
    if pokazat_zapros:
        print(f"→ Уходит модели {len(soobshcheniya)} сообщений:")
        for s in soobshcheniya:
            tekst = s["content"].replace("\n", " ")
            print(f"   {s['role']:<9} {len(tekst):>5} симв. | {tekst[:70]}{'…' if len(tekst) > 70 else ''}")
    otvet = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=max_tokens,
        messages=soobshcheniya, **dop,
    )
    return (otvet.choices[0].message.content or "").strip(), otvet.usage


PRAVILA = {"role": "system", "content": "Ты помощник школы. Отвечай по-русски, коротко: одно-два предложения."}

tekst, usage = sprosit([PRAVILA, {"role": "user", "content": "Привет!"}], pokazat_zapros=True)
print("\n🤖", tekst)
print(f"\nprompt_tokens = {usage.prompt_tokens}, completion_tokens = {usage.completion_tokens}")

Посмотри на `prompt_tokens`: даже на «Привет!» модель прочитала больше десятка токенов.
Это правила бота, служебная разметка ролей и само слово. Правила едут в **каждом**
запросе — запомни это, пригодится в шаге 7.

## Шаг 1. Модель правда ничего не помнит

Сделаем два **отдельных** запроса. В первом представимся, во втором спросим, как нас зовут.

In [ ]:
print("=== Запрос 1 ===")
tekst, _ = sprosit([PRAVILA, {"role": "user", "content": "Меня зовут Аня, я учусь в 9Б."}], pokazat_zapros=True)
print("🤖", tekst)

print("\n=== Запрос 2 — отдельный, без истории ===")
tekst, _ = sprosit([PRAVILA, {"role": "user", "content": "Как меня зовут?"}], pokazat_zapros=True)
print("🤖", tekst)

Во втором запросе модель не знает имени — ей просто неоткуда его взять. Первый запрос
для неё не существует.

А теперь то, что делает любой чат: **перешлём историю** вместе с новым вопросом.

In [ ]:
istoriya = [
    PRAVILA,
    {"role": "user", "content": "Меня зовут Аня, я учусь в 9Б."},
    {"role": "assistant", "content": "Приятно познакомиться, Аня!"},
    {"role": "user", "content": "Как меня зовут?"},
]
tekst, usage = sprosit(istoriya, pokazat_zapros=True)
print("🤖", tekst)
print(f"prompt_tokens = {usage.prompt_tokens}")

Теперь имя на месте — потому что оно **есть в запросе**. Это и есть вся «память» чата:

> **Модель без состояния** — модель, которая не хранит ничего между запросами: всё, что
> она должна «помнить», нужно положить в сам запрос.

Обрати внимание, что `prompt_tokens` вырос: мы заплатили за повторную пересылку
знакомства. На одном ходе это мелочь. Посмотрим, что будет на десяти.

## Шаг 2. Разговор из 10 ходов с полной историей

Вот сценарий. Он специально устроен как настоящий разговор:

* **ход 1** — важные факты: имя и то, что у Ани аллергия на орехи;
* **ход 5** — ученица вставляет длинный текст (как будто это результат инструмента или
  скопированная страница) — посмотрим, как он раздует историю;
* **ходы 9 и 10** — проверка памяти: помнит ли бот имя и аллергию.

In [ ]:
DLINNYY_TEKST = "Расписание столовой: " + "понедельник — каша, суп, котлета, компот; " * 25

SCENARIY = [
    "Привет! Меня зовут Аня, я из 9Б. И сразу: у меня аллергия на орехи.",
    "Во сколько обычно начинается первый урок?",
    "А сколько длится перемена?",
    "Что взять на экскурсию в музей?",
    "Вот расписание столовой, посмотри: " + DLINNYY_TEKST,
    "Какой день там самый сытный?",
    "Посоветуй, как готовиться к контрольной по алгебре.",
    "А если осталось всего два дня?",
    "Кстати, как меня зовут?",
    "Что мне нельзя есть? Предложи перекус на перемену.",
]

print(f"Ходов: {len(SCENARIY)}, длина хода 5: {len(SCENARIY[4])} символов")


def pomnit(otvety):
    """Проверка памяти по двум последним ответам: имя и аллергия.

    Проверяем кодом, а не на глаз (тема 5). «Орехи» засчитываем, только если бот
    говорит, что их нельзя, — бот без памяти может как раз посоветовать орехи.
    """
    imya, pro_orehi = otvety[-2].lower(), otvety[-1].lower()
    # «Вам нельзя есть продукты с орехами» — между «нельзя» и «орех» бывают другие слова,
    # поэтому ищем их порознь: орехи упомянуты И рядом есть запрет или аллергия.
    zapret = re.search(r"аллерг|нельзя|избега|исключ|без орех|не ешь|не стоит|опасн", pro_orehi)
    return {
        "имя": bool(re.search(r"\bан[яеюи]\b", imya)),
        "орехи": "орех" in pro_orehi and bool(zapret),
    }


def polosa(chislo, maksimum, shirina=40):
    return "█" * max(1, round(chislo / maksimum * shirina))


def progon(nazvanie, sobrat_zapros, sluzhebnye=None):
    """Проводит весь разговор.

    sobrat_zapros(istoriya) решает, ЧТО из истории отправить модели.
    sluzhebnye — список, куда стратегия пишет токены своих дополнительных запросов.
    """
    print(f"\n{'=' * 78}\n{nazvanie}\n{'=' * 78}")
    istoriya, otvety, po_hodam = [], [], []
    for nomer, vopros in enumerate(SCENARIY, 1):
        istoriya.append({"role": "user", "content": vopros})
        zapros = [PRAVILA] + sobrat_zapros(istoriya)
        tekst, usage = sprosit(zapros)
        istoriya.append({"role": "assistant", "content": tekst})
        otvety.append(tekst)
        po_hodam.append((nomer, len(zapros), usage.prompt_tokens, usage.completion_tokens))

    maks = max(p for _, _, p, _ in po_hodam)
    print(f"{'ход':>3} {'сообщ.':>6} {'вход':>6} {'выход':>6}  рост запроса")
    for nomer, soobsh, vhod, vyhod in po_hodam:
        print(f"{nomer:>3} {soobsh:>6} {vhod:>6} {vyhod:>6}  {polosa(vhod, maks)}")

    vsego_vhod = sum(p for _, _, p, _ in po_hodam)
    vsego_vyhod = sum(c for _, _, _, c in po_hodam)
    dop = sum(sluzhebnye or [])
    print(f"\nВход за разговор: {vsego_vhod} токенов"
          + (f" + служебные запросы: {dop}" if dop else ""))
    print(f"Выход за разговор: {vsego_vyhod} токенов")
    print(f"\n❓ {SCENARIY[-2]}\n🤖 {otvety[-2]}")
    print(f"❓ {SCENARIY[-1]}\n🤖 {otvety[-1]}")
    proverka = pomnit(otvety)
    print(f"\nПомнит имя: {'✅' if proverka['имя'] else '❌'}   помнит про орехи: {'✅' if proverka['орехи'] else '❌'}")
    return {"вход": vsego_vhod + dop, "выход": vsego_vyhod, **proverka}


ITOGI = {}
ITOGI["полная история"] = progon("ПОЛНАЯ ИСТОРИЯ", lambda istoriya: istoriya)

**Что посмотреть в выводе:**

1. Колонка **«вход»** растёт на каждом ходе — даже когда вопрос короткий. Каждый ход
   заново оплачивает всё сказанное раньше.
2. На **ходе 5** полоса резко прыгает: длинное расписание. И — самое важное — она
   **не опускается обратно** на ходах 6–10. Расписание нужно было один раз, а платим за
   него до конца разговора.
3. Бот, скорее всего, помнит и имя, и аллергию: всё же на месте.

Запомни число «Вход за разговор». Дальше будем с ним сравнивать.

## Шаг 3. Скользящее окно

Отправляем только последние `OKNO` сообщений. Одна строчка: `istoriya[-OKNO:]`.

> **Скользящее окно** — модели отправляются только несколько последних сообщений;
> более старые не отправляются вовсе.

In [ ]:
OKNO = 4   # 4 сообщения = 2 последних хода (вопрос + ответ)

ITOGI["окно"] = progon(f"СКОЛЬЗЯЩЕЕ ОКНО: последние {OKNO} сообщений", lambda istoriya: istoriya[-OKNO:])

**Что посмотреть в выводе:**

1. Колонка **«сообщ.»** перестала расти после третьего хода — окно заполнилось.
2. Полоса после хода 5 **опускается**: расписание выпало из окна через два хода.
3. Вход за разговор заметно меньше, чем в шаге 2.
4. А теперь проверка памяти. Знакомство было в ходе 1 — оно выпало из окна давным-давно.
   При подготовке лаборатории бот ответил «Я не знаю вашего имени», а на вопрос, что ей
   нельзя есть, посоветовал избегать сладостей и фастфуда — **про орехи ни слова**.

Вот цена окна: оно не предупреждает, что что-то забыло.

### Вариант: окно побольше

Может, дело просто в размере? Попробуем окно в 8 сообщений. Запусти и сравни вход
и проверку памяти. Помнит ли бот теперь то, что было на ходе 1?

In [ ]:
progon("ОКНО: 8 сообщений", lambda istoriya: istoriya[-8:])

Окно побольше стоит дороже и **всё равно** забывает начало, просто позже. Если важное
сказано в начале, увеличивать окно бесполезно — нужен другой приём.

## Шаг 4. Пересказ старого

Идея: когда история становится длинной, старую часть отдаём модели отдельным запросом
«перескажи коротко» и дальше отправляем **пересказ + последние сообщения**.

Сначала посмотрим на сам пересказ отдельно, чтобы понимать, что именно получает бот.

In [ ]:
PRAVILO_PERESKAZA = (
    "Перескажи разговор ниже в 2–3 предложениях. Обязательно сохрани: имя собеседника, "
    "его класс, любые ограничения и просьбы (например, аллергии). Пропусти мелкие детали "
    "и длинные вставленные тексты."
)


def pereskazat(soobshcheniya, predydushchiy=""):
    tekst = "\n".join(f"{s['role']}: {s['content'][:400]}" for s in soobshcheniya)
    if predydushchiy:
        tekst = f"Прошлый пересказ: {predydushchiy}\n\nНовые сообщения:\n{tekst}"
    otvet, usage = sprosit([{"role": "system", "content": PRAVILO_PERESKAZA},
                            {"role": "user", "content": tekst}], max_tokens=200)
    return otvet, usage.prompt_tokens + usage.completion_tokens


# Пробный пересказ первых трёх ходов разговора
proba = [{"role": "user", "content": SCENARIY[0]}, {"role": "assistant", "content": "Привет, Аня! Учту."},
         {"role": "user", "content": SCENARIY[1]}, {"role": "assistant", "content": "Обычно в 8:30."},
         {"role": "user", "content": SCENARIY[2]}, {"role": "assistant", "content": "Обычно 10 минут."}]
tekst, tokenov = pereskazat(proba)
print("Пересказ:", tekst)
print(f"Стоил токенов: {tokenov}")

Прочитай пересказ внимательно. Есть ли там имя? Аллергия? Модель пишет его сама —
и может что-то упустить. Мы специально попросили «обязательно сохрани ограничения»:
без этой фразы аллергия часто пропадает.

Теперь встроим пересказ в разговор. Когда за краем окна накопилось `PACHKA` сообщений,
сжимаем их в пересказ. Пересказ **накопительный**: новый строится из прошлого пересказа
и выпавших сообщений — так не нужно пересказывать весь разговор каждый раз заново.

Почему пачкой: при подготовке лаборатории первая версия пересказывала на каждом ходе —
и 7 служебных запросов съели почти всю экономию. Каждый ход из окна выпадает два
сообщения, и пересказывать их по одному — всё равно что возвращаться домой за каждой
забытой вещью отдельно.

In [ ]:
PACHKA = 6          # пересказываем, когда за краем окна накопилось столько сообщений
sluzhebnye_pereskaz = []
sostoyanie = {"pereskaz": "", "szhato_do": 0}


def s_pereskazom(istoriya):
    granica = len(istoriya) - OKNO
    if granica - sostoyanie["szhato_do"] >= PACHKA:
        novye = istoriya[sostoyanie["szhato_do"]:granica]
        sostoyanie["pereskaz"], tokenov = pereskazat(novye, sostoyanie["pereskaz"])
        sostoyanie["szhato_do"] = granica
        sluzhebnye_pereskaz.append(tokenov)
        print(f"   [пересказ обновлён, {tokenov} токенов]: {sostoyanie['pereskaz'][:110]}…")
    zapros = istoriya[sostoyanie["szhato_do"]:]
    if sostoyanie["pereskaz"]:
        zapros = [{"role": "system", "content": "Кратко о начале разговора: " + sostoyanie["pereskaz"]}] + zapros
    return zapros


ITOGI["пересказ"] = progon("ОКНО + ПЕРЕСКАЗ", s_pereskazom, sluzhebnye_pereskaz)
print("\nИтоговый пересказ, который видел бот в конце:\n", sostoyanie["pereskaz"])

**Что посмотреть в выводе:**

1. Строки **«[пересказ обновлён]»** появляются не на каждом ходе, а когда за краем окна
   накопилась пачка. Каждая такая строка — отдельный запрос, и его токены посчитаны
   в «служебных». Поставь `PACHKA = 2` и посмотри, во что обойдётся пересказ на каждом
   ходе: при подготовке лаборатории вышло 7 служебных запросов на 1429 токенов.
2. Вход за разговор (со служебными) — между окном и полной историей.
3. Проверка памяти: имя и аллергия должны быть на месте — если модель не потеряла их
   при пересказе. При подготовке лаборатории пересказ сохранил и то и другое: «Аня из
   9Б класса, ранее сообщившая об аллергии на орехи…». Посмотри итоговый пересказ:
   что в нём сохранилось у тебя?
4. Расписание столовой в пересказ почти наверняка не попало — и правильно: мы попросили
   пропускать длинные вставки.

## Шаг 5. Долгая память: факты отдельно

Пересказ доверяет модели решать, что важно. Можно поступить строже: на каждом ходе
просить модель **выписать факты** о собеседнике в JSON и хранить их отдельным списком.
Список короткий и подставляется в каждый запрос, а история живёт в маленьком окне.

Здесь пригодится схема ответа из темы 1: нам нужен не текст, а список, который читает программа.

In [ ]:
PRAVILO_FAKTOV = (
    "Из сообщения пользователя выпиши устойчивые факты о нём: имя, класс, ограничения, "
    "постоянные просьбы. Не выписывай вопросы и временное. "
    'Ответь JSON вида {"fakty": ["...", "..."]}. Если фактов нет — {"fakty": []}.'
)


def vypisat_fakty(soobshchenie):
    otvet, usage = sprosit(
        [{"role": "system", "content": PRAVILO_FAKTOV}, {"role": "user", "content": soobshchenie[:500]}],
        max_tokens=100, response_format={"type": "json_object"},
    )
    try:
        fakty = json.loads(otvet).get("fakty", [])
    except (json.JSONDecodeError, AttributeError):
        fakty = []            # модель ответила не JSON — ничего не запоминаем, но и не падаем
    return fakty, usage.prompt_tokens + usage.completion_tokens


# Проверим на первых двух сообщениях сценария
for soobshchenie in SCENARIY[:2]:
    fakty, tokenov = vypisat_fakty(soobshchenie)
    print(f"«{soobshchenie[:50]}…» → {fakty}  ({tokenov} токенов)")

Из первого сообщения должны выписаться имя, класс и аллергия, из второго — ничего:
вопрос про уроки не факт о человеке. Если модель выписала что-то лишнее — это та самая
ненадёжность из темы 1. В настоящем проекте такой список стоит время от времени
просматривать глазами.

In [ ]:
sluzhebnye_pamyat = []
pamyat = []


def s_pamyatyu(istoriya):
    posledneye = istoriya[-1]["content"]
    fakty, tokenov = vypisat_fakty(posledneye)
    sluzhebnye_pamyat.append(tokenov)
    for fakt in fakty:
        if fakt not in pamyat:
            pamyat.append(fakt)
            print(f"   [в память: {fakt}]")
    zapros = istoriya[-OKNO:]
    if pamyat:
        zapros = [{"role": "system", "content": "Известно о собеседнике: " + "; ".join(pamyat)}] + zapros
    return zapros


ITOGI["долгая память"] = progon("ОКНО + ДОЛГАЯ ПАМЯТЬ", s_pamyatyu, sluzhebnye_pamyat)
print("\nЧто лежит в долгой памяти:")
pprint(pamyat)

**Что посмотреть в выводе:**

1. Строки **«[в память: …]»** появились в основном на первом ходе. Остальные ходы
   фактов не добавили — и хорошо.
2. Служебных токенов здесь обычно **больше**, чем у пересказа: мы проверяем на факты каждое
   сообщение. Это плата за надёжность. В жизни экономят так: проверяют только
   сообщения пользователя и только короткие.
3. Проверка памяти: имя и аллергия должны быть на месте, причём не благодаря удаче
   пересказа, а потому что лежат отдельной строкой в каждом запросе.

## Шаг 6. Сравнение

Соберём итоги всех способов. «Вход» здесь — со служебными запросами, то есть честная
полная цена чтения.

In [ ]:
maks = max(v["вход"] for v in ITOGI.values())
print(f"{'способ':<16} {'вход':>7} {'выход':>6}  имя  орехи  вход относительно полной истории")
for nazvanie, v in ITOGI.items():
    dolya = v["вход"] / ITOGI["полная история"]["вход"]
    print(f"{nazvanie:<16} {v['вход']:>7} {v['выход']:>6}   {'✅' if v['имя'] else '❌'}   {'✅' if v['орехи'] else '❌'}   "
          f"{polosa(v['вход'], maks, 30)} {dolya:.0%}")

CENA_VHODA = 0.03   # qwen3.7-flash: $ за 1 млн входных токенов
print(f"\nЕсли таких разговоров 10 000 в месяц, по цене входа ${CENA_VHODA} за 1 млн:")
for nazvanie, v in ITOGI.items():
    print(f"  {nazvanie:<16} ${v['вход'] * 10_000 * CENA_VHODA / 1e6:.2f}")

Как читать эту таблицу:

* **Окно** почти всегда самое дешёвое — и почти всегда проваливает проверку памяти.
  Дешевизна, которая советует орехи человеку с аллергией, не нужна.
* **Пересказ** и **долгая память** стоят больше окна из-за служебных запросов, но
  помнят важное.
* При подготовке лаборатории вышло: полная история — 4665 токенов входа, окно — 1848
  (и забыло всё), пересказ — 3989, долгая память — 3329 (обе помнят). Пересказ оказался
  всего на 14% дешевле полной истории — служебные запросы съели почти всю экономию.
* На **10 ходах** выгода от экономии скромная. На 50 и 100 ходах разница становится
  огромной: полная история растёт с каждым ходом, остальные способы — почти нет.
  Проверь это в задании 1 ниже.

Цены здесь маленькие, потому что модель дешёвая. У сильной модели они умножаются
в 30–100 раз — посмотри цены на витрине моделей.

## Шаг 7. Кэш начала запроса

Есть часть запроса, которую не сократишь, — правила бота. У настоящих ботов они длинные:
описание школы, стиль ответов, запреты. И они **одинаковые** в каждом запросе.

Многие поставщики запоминают начало запроса. Если следующий запрос начинается точно
так же, это начало стоит дешевле. Сервер сообщает, сколько токенов взято из кэша,
в поле `usage.prompt_tokens_details.cached_tokens`.

Проверим: отправим два запроса с одинаковыми длинными правилами и разными вопросами.

In [ ]:
DLINNYE_PRAVILA = {"role": "system", "content": (
    "Ты помощник школы №7. " + "Отвечай вежливо, коротко, по-русски, без выдумок и лишних деталей. " * 60
)}


def pokazat_kesh(nazvanie, soobshcheniya):
    otvet = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=30, messages=soobshcheniya)
    usage = otvet.usage
    detali = getattr(usage, "prompt_tokens_details", None)
    kesh = getattr(detali, "cached_tokens", None) if detali else None
    print(f"{nazvanie:<34} prompt_tokens={usage.prompt_tokens:>5}  из кэша: {kesh if kesh is not None else 'сервер не сообщил'}"
          f"  (ответила {otvet.model})")


pokazat_kesh("1) правила + вопрос про обед", [DLINNYE_PRAVILA, {"role": "user", "content": "Во сколько обед?"}])
pokazat_kesh("2) те же правила + другой вопрос", [DLINNYE_PRAVILA, {"role": "user", "content": "Когда каникулы?"}])
pokazat_kesh("3) время В НАЧАЛЕ + те же правила",
             [{"role": "system", "content": "Сейчас 15:42:07. " + DLINNYE_PRAVILA["content"]},
              {"role": "user", "content": "Когда каникулы?"}])

**Как читать результат — честно, потому что здесь многое зависит не от тебя:**

* Если во втором запросе «из кэша» больше нуля, а в третьем снова ноль или меньше —
  ты увидел кэш своими глазами: одинаковое начало узнано, а одна строка времени в самом
  начале сломала совпадение. При подготовке лаборатории так и вышло: во втором запросе
  **1280 токенов из 1411** взяты из кэша, а в третьем — **0**, хотя правила те же самые.
* Если везде «0» или «сервер не сообщил» — это тоже нормальный результат. Кэш включается
  не у всех поставщиков, часто только для запросов длиннее 1000–2000 токенов, и наш
  школьный сервер может переключиться на другую модель (посмотри, какая ответила).

Правило от этого не меняется: **неизменное — в начало, меняющееся — в конец**. Даже там,
где кэша сегодня нет, так устроенный запрос ничего не теряет, а там, где есть, — экономит.

## Попробуй сам

1. **Длинный разговор.** Добавь в `SCENARIY` ещё 10 вопросов перед двумя последними
   и перезапусти шаги 2 и 3. Во сколько раз теперь полная история дороже окна? Было ли
   так же на 10 ходах?
2. **Сломай пересказ.** Убери из `PRAVILO_PERESKAZA` фразу про ограничения и аллергии
   и перезапусти шаг 4. Сохранилась ли аллергия в пересказе?
3. **Обрезка вставок.** Сделай стратегию, которая в истории режет длинные сообщения
   пользователя до 200 символов: `[{**s, "content": s["content"][:200]} for s in istoriya]`.
   Как изменился вход на ходах 6–10? Помнит ли бот, что в расписании?
4. **Своя проверка.** Добавь в сценарий на ходе 3 «я не ем мясо» и в конце спроси
   «что мне можно на обед?». Какие способы это помнят?

## Что унести с собой

* Модель ничего не помнит: всё, что бот «знает» о разговоре, лежит в запросе.
* С полной историей каждый ход оплачивает все предыдущие, а длинная вставка остаётся
  в счёте до конца разговора.
* Окно дешёвое, но молча забывает начало — а в начале обычно самое важное.
* Пересказ и долгая память стоят служебных запросов, зато сохраняют важное.
* Неизменное — в начало запроса, меняющееся — в конец: это бесплатная экономия.